# BIRL Step 05 — SVI

In [ ]:
# ============================================================
# Cell 0: Colab Setup
# ============================================================
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'numpyro>=0.15', 'pyarrow>=14', 'pandas>=2.1', 'scipy>=1.12'],
    capture_output=True)

import os, shutil
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/BIRL_SVI'
os.makedirs(DRIVE_DIR, exist_ok=True)
WORK_DIR = '/content'
os.chdir(WORK_DIR)

for fname in ['birl_sample.parquet', 'env_model_output.npz', 'action_space_config.json']:
    dst = os.path.join(WORK_DIR, fname)
    src = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(dst):
        print(f'{fname} already local')
    elif os.path.exists(src):
        shutil.copy2(src, dst); print(f'Copied {fname} from Drive')
    else:
        r = subprocess.run(['gsutil','-q','cp',f'gs://subsahra/birl/data/{fname}',dst],
                           capture_output=True, timeout=120)
        if r.returncode == 0:
            print(f'Downloaded {fname} from GCS'); shutil.copy2(dst, src)
        else:
            print(f'WARNING: {fname} not found!')

import jax
print(f'JAX {jax.__version__} | {jax.devices()[0]}')

In [ ]:
# ============================================================
# Cell 1: Imports
# ============================================================
import warnings; warnings.filterwarnings('ignore')
import json, time, pickle, os, shutil
from datetime import datetime
import numpy as np
import pandas as pd
import jax, jax.numpy as jnp
import numpyro, numpyro.distributions as dist
from numpyro import plate, sample, deterministic
from numpyro.infer import SVI, Trace_ELBO, Predictive
from numpyro.infer.autoguide import AutoNormal
from numpyro.optim import ClippedAdam
import matplotlib.pyplot as plt

def save_drive(path):
    shutil.copy2(path, os.path.join(DRIVE_DIR, os.path.basename(path)))

def load_drive(fname):
    s = os.path.join(DRIVE_DIR, fname)
    if os.path.exists(s): shutil.copy2(s, os.path.join(WORK_DIR, fname)); return True
    return False

print(f'NumPyro {numpyro.__version__}')

In [ ]:
# ============================================================
# Cell 2: Data Preparation (run once, shared by all variants)
# ============================================================
t0 = time.time()
df = pd.read_parquet('birl_sample.parquet')
env = np.load('env_model_output.npz', allow_pickle=True)
with open('action_space_config.json') as f:
    action_cfg = json.load(f)
N_ACTIONS = action_cfg['n_actions']

hh_list = np.sort(df['hh_id_merge'].unique())
hh_to_idx = {h: i for i, h in enumerate(hh_list)}
df['hh_idx'] = df['hh_id_merge'].map(hh_to_idx).astype(np.int32)
countries = sorted(df['country'].unique())
df['country_idx'] = df['country'].map({c:i for i,c in enumerate(countries)}).astype(np.int32)
hh_info = df.drop_duplicates('hh_id_merge').set_index('hh_idx').sort_index()
hh_countries = hh_info['country'].values

df['cz_key'] = df['country'] + '_' + df['action_zone'].astype(str)
cz_keys = sorted(df['cz_key'].unique())
cz_map = {k:i for i,k in enumerate(cz_keys)}
df['cz_idx'] = df['cz_key'].map(cz_map).astype(np.int32)

N_OBS, N_HH, N_COUNTRY, N_CZ = len(df), len(hh_list), len(countries), len(cz_keys)

feas = np.zeros((N_CZ, N_ACTIONS), dtype=bool)
for (ck,aid),_ in df.groupby(['cz_key','action_id']).size().items():
    feas[cz_map[ck], aid] = True
assert np.sum(~feas[df['cz_idx'].values, df['action_id'].values]) == 0

_e = 0.01
q10s = np.maximum(np.nan_to_num(env['q10'],nan=_e),_e).astype(np.float32)
q50s = np.maximum(np.nan_to_num(env['q50'],nan=_e),_e).astype(np.float32)
q90s = np.maximum(np.nan_to_num(env['q90'],nan=_e),_e).astype(np.float32)
slg = np.clip(np.log(np.maximum(q90s/q50s,1.0001))/1.282, 0.01, 5.0).astype(np.float32)

# Full dataset JAX arrays
D = dict(
    obs_action=jnp.array(df['action_id'].values, dtype=jnp.int32),
    obs_hh_idx=jnp.array(df['hh_idx'].values, dtype=jnp.int32),
    hh_country_idx=jnp.array(hh_info['country_idx'].values, dtype=jnp.int32),
    obs_cz_idx=jnp.array(df['cz_idx'].values, dtype=jnp.int32),
    feasibility_mask=jnp.array(feas),
    cf_log_q10=jnp.array(np.log(q10s)), cf_log_q50=jnp.array(np.log(q50s)),
    cf_log_q90=jnp.array(np.log(q90s)), cf_sigma_log=jnp.array(slg),
    N_hh=N_HH, N_country=N_COUNTRY, N_obs=N_OBS, batch_size=4096)

# R1 extras
D_r1_extra = dict(
    survival_threshold=jnp.array(df['survival_threshold_P25'].fillna(0).values, dtype=jnp.float32),
    min_consumption=jnp.array(df['totcons_USD'].fillna(0).values, dtype=jnp.float32))

# R4 fixed gamma
hh_gamma_fixed = jnp.array(
    df.groupby('hh_idx')['survival_threshold_P25'].median().sort_index().fillna(20.0).values,
    dtype=jnp.float32)

# R9 subset
r9_order = df['harvest_value_USD'].fillna(0).values.argsort()[::-1]
r9_seen, r9_keep = set(), []
for i in r9_order:
    did = df.iloc[i]['decision_id']
    if did not in r9_seen: r9_seen.add(did); r9_keep.append(i)
r9_idx = np.sort(np.array(r9_keep))
r9_df = df.iloc[r9_idx].reset_index(drop=True)
r9_hh = np.sort(r9_df['hh_id_merge'].unique())
r9_hh_map = {h:i for i,h in enumerate(r9_hh)}
r9_df['hh_idx'] = r9_df['hh_id_merge'].map(r9_hh_map).astype(np.int32)
r9_hi = r9_df.drop_duplicates('hh_id_merge').set_index('hh_idx').sort_index()
r9_df['cz_key'] = r9_df['country'] + '_' + r9_df['action_zone'].astype(str)
r9_czk = sorted(r9_df['cz_key'].unique())
r9_czm = {k:i for i,k in enumerate(r9_czk)}
r9_df['cz_idx'] = r9_df['cz_key'].map(r9_czm).astype(np.int32)
r9_feas = np.zeros((len(r9_czk), N_ACTIONS), dtype=bool)
for (ck,aid),_ in r9_df.groupby(['cz_key','action_id']).size().items():
    r9_feas[r9_czm[ck], aid] = True
r9_q10 = np.maximum(np.nan_to_num(np.load('env_model_output.npz')['q10'][r9_idx],nan=_e),_e).astype(np.float32)
r9_q50 = np.maximum(np.nan_to_num(np.load('env_model_output.npz')['q50'][r9_idx],nan=_e),_e).astype(np.float32)
r9_q90 = np.maximum(np.nan_to_num(np.load('env_model_output.npz')['q90'][r9_idx],nan=_e),_e).astype(np.float32)
D_r9 = dict(
    obs_action=jnp.array(r9_df['action_id'].values, dtype=jnp.int32),
    obs_hh_idx=jnp.array(r9_df['hh_idx'].values, dtype=jnp.int32),
    hh_country_idx=jnp.array(r9_hi['country_idx'].values, dtype=jnp.int32),
    obs_cz_idx=jnp.array(r9_df['cz_idx'].values, dtype=jnp.int32),
    feasibility_mask=jnp.array(r9_feas),
    cf_log_q10=jnp.array(np.log(r9_q10)), cf_log_q50=jnp.array(np.log(r9_q50)),
    cf_log_q90=jnp.array(np.log(r9_q90)),
    cf_sigma_log=jnp.array(np.clip(np.log(np.maximum(r9_q90/r9_q50,1.0001))/1.282,0.01,5.0).astype(np.float32)),
    N_hh=len(r9_hh), N_country=N_COUNTRY, N_obs=len(r9_df), batch_size=4096)

del env, q10s, q50s, q90s, slg
print(f'N_obs={N_OBS:,} N_hh={N_HH:,} N_cz={N_CZ} R9={len(r9_idx):,} ({time.time()-t0:.1f}s)')

In [ ]:
# ============================================================
# Cell 3: Shared utility function + z-score + run_variant()
# ============================================================

from functools import partial
from tqdm.auto import tqdm

def _safe_exp(x, cap=20.0):
    return jnp.exp(jnp.clip(x, -cap, cap))


def stone_geary_utility(Y, gamma, rho, eps=1.0):
    surplus = jnp.maximum(Y - gamma, eps)
    a = 1.0 - rho
    log_s = jnp.log(surplus)
    taylor = log_s * (1.0 + 0.5 * a * log_s)
    safe_a = jnp.where(jnp.abs(a) < 0.02, 1.0, a)
    exponent = jnp.clip(safe_a * log_s, -20.0, 20.0)
    power = (jnp.exp(exponent) - 1.0) / safe_a
    return jnp.where(jnp.abs(a) < 0.02, taylor, power)


def _zscore_reward(reward, mask):
    """Center rewards over feasible actions (mean-subtract only).
    Preserves the natural scale of utility differences so that
    beta and rho/gamma remain identifiable."""
    n_feasible = mask.sum(axis=-1, keepdims=True).clip(min=1)
    r_masked = jnp.where(mask, reward, 0.0)
    r_mean = r_masked.sum(axis=-1, keepdims=True) / n_feasible
    return reward - r_mean


# 'beta' removed — it's now learned inside the model
_STATIC_KEYS = frozenset({
    'N_hh', 'N_country', 'N_obs', 'batch_size',
    'eps', 'fixed_rho', 'fixed_gamma',
})


def run_variant(name, model_fn, kwargs, total_steps=40000, peak_lr=0.003):
    print(f'\n{"="*60}')
    print(f'  VARIANT: {name}')
    print(f'{"="*60}')
    prefix = f'{name}_'
    ckpt_file = f'{prefix}svi_checkpoint.pkl'
    WARMUP, MIN_LR, CKPT_EVERY, LOG_EVERY = 3000, 0.0003, 5000, 200

    static_kw = {k: v for k, v in kwargs.items() if k in _STATIC_KEYS}
    dyn_kw    = {k: v for k, v in kwargs.items() if k not in _STATIC_KEYS}
    bound_model = partial(model_fn, **static_kw)

    def lr_sched(step):
        w = peak_lr * jnp.minimum(step / WARMUP, 1.0)
        d = jnp.maximum(step - WARMUP, 0)
        c = MIN_LR + 0.5 * (peak_lr - MIN_LR) * (1 + jnp.cos(
            jnp.pi * jnp.minimum(d / (total_steps - WARMUP), 1.0)))
        return jnp.where(step < WARMUP, w, c)

    guide = AutoNormal(bound_model)
    opt   = ClippedAdam(step_size=lr_sched, clip_norm=2.0)
    svi   = SVI(bound_model, guide, opt, Trace_ELBO())

    svi_state = svi.init(jax.random.PRNGKey(42), **dyn_kw)
    init_elbo = -float(svi.evaluate(svi_state, **dyn_kw))
    n_params  = sum(np.array(v).size for v in svi.get_params(svi_state).values())
    print(f'Init ELBO: {init_elbo:.1f} | Guide params: {n_params:,}')

    start, hist = 0, []
    if load_drive(ckpt_file):
        with open(ckpt_file, 'rb') as f:
            p = pickle.load(f)
        last_elbo = p['elbo_history'][-1] if p['elbo_history'] else float('nan')
        if np.isnan(last_elbo):
            print(f'Checkpoint is NaN-corrupted — ignoring, starting fresh.')
        else:
            svi_state = jax.tree.unflatten(
                p['treedef'], [jnp.array(x) for x in p['flat_state']])
            start, hist = p['step'], p['elbo_history']
            print(f'Resumed: step={start}, ELBO={hist[-1]:.1f}')

    upd = jax.jit(svi.update)
    t0 = time.time()
    rl = rc = 0
    best = max(hist) if hist else -1e18
    consecutive_nan = 0
    MAX_CONSECUTIVE_NAN = 10

    pbar = tqdm(range(start, total_steps),
                initial=start, total=total_steps,
                desc=name, unit='step',
                bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} '
                           '[{elapsed}<{remaining}, {rate_fmt}] {postfix}')

    for step in pbar:
        svi_state, loss = upd(svi_state, **dyn_kw)

        if jnp.isnan(loss):
            consecutive_nan += 1
            if consecutive_nan >= MAX_CONSECUTIVE_NAN:
                pbar.write(f'NaN x{consecutive_nan} consecutive — giving up.')
                break
            pbar.write(f'NaN at step {step+1} — reinitializing '
                       f'({consecutive_nan}/{MAX_CONSECUTIVE_NAN})...')
            svi_state = svi.init(jax.random.PRNGKey(step * 7 + 137), **dyn_kw)
            upd = jax.jit(svi.update)
            continue

        consecutive_nan = 0
        e = -float(loss)
        hist.append(e)
        rl += float(loss)
        rc += 1
        if e > best:
            best = e

        if (step + 1) % LOG_EVERY == 0 and rc > 0:
            avg_elbo = -rl / rc
            pbar.set_postfix(ELBO=f'{avg_elbo:.1f}', best=f'{best:.1f}')
            rl = rc = 0

        if (step + 1) % CKPT_EVERY == 0:
            flat, td = jax.tree.flatten(svi_state)
            with open(ckpt_file + '.tmp', 'wb') as f:
                pickle.dump({
                    'flat_state': [np.array(x) for x in flat],
                    'treedef': td,
                    'step': step + 1,
                    'elbo_history': hist,
                }, f)
            os.rename(ckpt_file + '.tmp', ckpt_file)
            save_drive(ckpt_file)
            pbar.write(f'  checkpoint {step+1}')

    pbar.close()
    elapsed = (time.time() - t0) / 60
    final_elbo = hist[-1] if hist else float('nan')
    print(f'Done: {len(hist)} steps, {elapsed:.1f}min, ELBO={final_elbo:.1f}')

    flat, td = jax.tree.flatten(svi_state)
    with open(ckpt_file + '.tmp', 'wb') as f:
        pickle.dump({
            'flat_state': [np.array(x) for x in flat],
            'treedef': td,
            'step': len(hist),
            'elbo_history': hist,
        }, f)
    os.rename(ckpt_file + '.tmp', ckpt_file)
    save_drive(ckpt_file)
    pd.DataFrame({'step': range(1, len(hist) + 1), 'elbo': hist}).to_csv(
        f'{prefix}elbo.csv', index=False)
    save_drive(f'{prefix}elbo.csv')

    # -- Posterior --
    params = svi.get_params(svi_state)
    sites  = [k for k in ['alpha_i', 'rho_i', 'gamma_i', 'gamma_risk_i',
                           'lambda_i', 'beta']]
    pred   = Predictive(bound_model, guide=guide, params=params,
                        num_samples=1000, return_sites=sites)
    nps = {k: np.array(v)
           for k, v in pred(jax.random.PRNGKey(99), **dyn_kw).items()}

    cur_hh   = r9_hh if 'R9' in name else hh_list
    cur_ctry = (r9_df.drop_duplicates('hh_id_merge').sort_values('hh_idx')['country'].values
                if 'R9' in name else hh_countries)
    rows = {'hh_id_merge': cur_hh, 'country': cur_ctry}
    for p in ['rho', 'alpha', 'gamma', 'gamma_risk', 'lambda']:
        k = f'{p}_i'
        if k not in nps:
            continue
        s = nps[k]
        rows[f'{p}_mean'] = s.mean(0)
        rows[f'{p}_std']  = s.std(0)
        rows[f'{p}_q025'] = np.quantile(s, 0.025, axis=0)
        rows[f'{p}_q975'] = np.quantile(s, 0.975, axis=0)
    hh_df = pd.DataFrame(rows)
    hh_df.to_parquet(f'{prefix}hh_params.parquet', index=False)
    save_drive(f'{prefix}hh_params.parquet')

    np.savez_compressed(f'{prefix}guide.npz',
                        **{k: np.array(v) for k, v in params.items()})
    save_drive(f'{prefix}guide.npz')

    # -- Diagnostics --
    print(f'\nPosterior [{name}]:')
    if 'beta' in nps:
        bv = nps['beta']
        print(f'  beta (learned): mean={bv.mean():.3f}  std={bv.std():.3f}')
    for p, k, ex in [('gamma', 'gamma_i', '$5-100'),
                      ('rho',   'rho_i',   '0.5-3'),
                      ('alpha', 'alpha_i',  '-0.5~0.5')]:
        if k in nps:
            v = nps[k].mean(0)
            print(f'  {p}: med={np.median(v):.3f}  '
                  f'P5={np.percentile(v,5):.3f}  '
                  f'P95={np.percentile(v,95):.3f}  ({ex})')

    # ELBO plot
    if hist:
        fig, ax = plt.subplots(1, 2, figsize=(12, 4))
        ax[0].plot(hist, lw=0.3, alpha=0.5)
        ax[0].set_title(f'ELBO [{name}]')
        w = min(500, max(1, len(hist) // 10))
        if w > 1:
            ax[0].plot(pd.Series(hist).rolling(w).mean(), lw=1.5, color='darkblue')
        s = len(hist) * 3 // 4
        ax[1].plot(range(s, len(hist)), hist[s:], lw=0.5)
        ax[1].set_title('Last 25%')
        plt.tight_layout()
        plt.savefig(f'{prefix}elbo.png', dpi=150)
        save_drive(f'{prefix}elbo.png')
        plt.show()

    # PPC
    pk = dict(dyn_kw)
    pk['obs_action'] = None
    pa = np.array(Predictive(
        bound_model, guide=guide, params=params, num_samples=200,
        return_sites=['obs_action']
    )(jax.random.PRNGKey(999), **pk)['obs_action'])
    oa = np.array(kwargs['obs_action'])
    ppc = [{
        'action': a,
        'label':  action_cfg['action_labels'].get(str(a), f'a{a}'),
        'pred':   float((pa == a).mean()),
        'obs':    float((oa == a).mean()),
        'diff':   abs(float((pa == a).mean()) - float((oa == a).mean())),
    } for a in range(N_ACTIONS)]
    pd.DataFrame(ppc).to_csv(f'{prefix}ppc.csv', index=False)
    save_drive(f'{prefix}ppc.csv')
    print(f'PPC max diff: {max(r["diff"] for r in ppc):.4f}')

    # Metrics
    m = {
        'variant': name, 'model': model_fn.__name__,
        'steps': len(hist), 'final_elbo': final_elbo, 'best_elbo': best,
        'N_obs': kwargs['N_obs'], 'N_hh': kwargs['N_hh'],
        'timestamp': datetime.now().isoformat(),
    }
    for p in ['rho', 'alpha', 'gamma']:
        if f'{p}_i' in nps:
            m[f'{p}_med'] = float(np.median(nps[f'{p}_i'].mean(0)))
    if 'beta' in nps:
        m['beta_mean'] = float(nps['beta'].mean())
    with open(f'{prefix}metrics.json', 'w') as f:
        json.dump(m, f, indent=2)
    save_drive(f'{prefix}metrics.json')
    print(f'\nAll saved to Drive with prefix: {prefix}*')
    return nps, hist


print('run_variant() ready. Now run any variant cell below.')

In [ ]:
# ============================================================
# Cell 4: MAIN — Stone-Geary (α, ρ, γ) with bounded params
#          + learnable beta
# ============================================================

# --- Clean up old checkpoint ---
_ckpt = 'main_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

# --- Parameter bounds ---
RHO_LO, RHO_HI      = 0.1, 5.0
ALPHA_BOUND          = 1.5
GAMMA_LO, GAMMA_HI  = 1.0, 30.0

def model_main(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # ========== Learnable beta ==========
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))   # prior mean ≈ e^1 ≈ 2.7
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))
    #                                    clip → beta ∈ [0.14, 54.6]

    # ========== Global hyperpriors ==========
    if fixed_rho is None:
        mu_rho_G  = sample('mu_rho_G', dist.Normal(0.0, 1.0))
        sig_rho_G = sample('sigma_rho_G', dist.HalfNormal(0.5))
    mu_a_G  = sample('mu_alpha_G', dist.Normal(0.0, 1.0))
    sig_a_G = sample('sigma_alpha_G', dist.HalfNormal(0.5))
    if fixed_gamma is None:
        mu_lg_G  = sample('mu_lgamma_G', dist.Normal(0.0, 1.0))
        sig_lg_G = sample('sigma_lgamma_G', dist.HalfNormal(0.5))

    # ========== Country level ==========
    with plate('countries', N_country):
        if fixed_rho is None:
            mu_rho_C_r = sample('mu_rho_C_raw', dist.Normal(0, 1))
            sig_rho_C  = sample('sigma_rho_C', dist.HalfNormal(0.3))
        mu_a_C_r = sample('mu_alpha_C_raw', dist.Normal(0, 1))
        sig_a_C  = sample('sigma_alpha_C', dist.HalfNormal(0.3))
        if fixed_gamma is None:
            mu_lg_C_r = sample('mu_lgamma_C_raw', dist.Normal(0, 1))
            sig_lg_C  = sample('sigma_lgamma_C', dist.HalfNormal(0.3))

    if fixed_rho is None:
        mu_rho_C = deterministic('mu_rho_C', mu_rho_G + sig_rho_G * mu_rho_C_r)
    mu_a_C = deterministic('mu_alpha_C', mu_a_G + sig_a_G * mu_a_C_r)
    if fixed_gamma is None:
        mu_lg_C = deterministic('mu_lgamma_C', mu_lg_G + sig_lg_G * mu_lg_C_r)

    # ========== Household level ==========
    with plate('households', N_hh):
        a_r = sample('alpha_raw', dist.Normal(0, 1))
        if fixed_rho is None:
            rho_r = sample('rho_raw', dist.Normal(0, 1))
        if fixed_gamma is None:
            lg_r = sample('lgamma_raw', dist.Normal(0, 1))

    # ========== Bounded transforms ==========
    alpha_unbounded = mu_a_C[hh_country_idx] + sig_a_C[hh_country_idx] * a_r
    alpha_i = deterministic('alpha_i', ALPHA_BOUND * jnp.tanh(alpha_unbounded))

    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_unbounded = mu_rho_C[hh_country_idx] + sig_rho_C[hh_country_idx] * rho_r
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        lg_unbounded = mu_lg_C[hh_country_idx] + sig_lg_C[hh_country_idx] * lg_r
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(lg_unbounded))

    # ========== Observation level ==========
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        ao = alpha_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        log_q10_adj = jnp.clip(cf_log_q10[idx] - ao * sl, -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx] - ao * sl, -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx] - ao * sl, -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


# ---------- Sanity check ----------
print("Running sanity check (1 step)...")
# Remove beta from D if present (now learned in model)
D.pop('beta', None)
_bound = partial(model_main, **{k: v for k, v in D.items() if k in _STATIC_KEYS})
_dyn   = {k: v for k, v in D.items() if k not in _STATIC_KEYS}
_guide = AutoNormal(_bound)
_svi   = SVI(_bound, _guide, ClippedAdam(step_size=1e-4, clip_norm=1.0), Trace_ELBO())
_state = _svi.init(jax.random.PRNGKey(0), **_dyn)
_state, _loss = jax.jit(_svi.update)(_state, **_dyn)
print(f"  1-step loss = {float(_loss):.1f}  {'OK' if not np.isnan(float(_loss)) else 'NaN!'}")
del _bound, _dyn, _guide, _svi, _state, _loss

# ---------- Full run ----------
nps_main, hist_main = run_variant('main', model_main, D)

In [ ]:
# 快速看一下数据尺度
q50_values = np.exp(np.array(D['cf_log_q50']))
print(f"Consumption (exp(cf_log_q50)):")
print(f"  median: {np.median(q50_values):.1f}")
print(f"  P5:     {np.percentile(q50_values, 5):.1f}")
print(f"  P95:    {np.percentile(q50_values, 95):.1f}")
print(f"  min:    {np.min(q50_values):.1f}")
print(f"  max:    {np.max(q50_values):.1f}")
print(f"\nshape: {q50_values.shape}")
print(f"cf_sigma_log range: [{float(D['cf_sigma_log'].min()):.3f}, {float(D['cf_sigma_log'].max()):.3f}]")

In [ ]:
# ============================================================
# Cell 4b: Comparison — model_main with alpha fixed to 0
# ============================================================

_ckpt = 'noalpha_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

RHO_LO, RHO_HI      = 0.1, 5.0
GAMMA_LO, GAMMA_HI  = 0.1, 30.0

def model_main_noalpha(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # ========== Learnable beta ==========
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # ========== Global hyperpriors (no alpha) ==========
    if fixed_rho is None:
        mu_rho_G  = sample('mu_rho_G', dist.Normal(0.0, 1.0))
        sig_rho_G = sample('sigma_rho_G', dist.HalfNormal(0.5))
    if fixed_gamma is None:
        mu_lg_G  = sample('mu_lgamma_G', dist.Normal(0.0, 1.0))
        sig_lg_G = sample('sigma_lgamma_G', dist.HalfNormal(0.5))

    # ========== Country level (no alpha) ==========
    with plate('countries', N_country):
        if fixed_rho is None:
            mu_rho_C_r = sample('mu_rho_C_raw', dist.Normal(0, 1))
            sig_rho_C  = sample('sigma_rho_C', dist.HalfNormal(0.3))
        if fixed_gamma is None:
            mu_lg_C_r = sample('mu_lgamma_C_raw', dist.Normal(0, 1))
            sig_lg_C  = sample('sigma_lgamma_C', dist.HalfNormal(0.3))

    if fixed_rho is None:
        mu_rho_C = deterministic('mu_rho_C', mu_rho_G + sig_rho_G * mu_rho_C_r)
    if fixed_gamma is None:
        mu_lg_C = deterministic('mu_lgamma_C', mu_lg_G + sig_lg_G * mu_lg_C_r)

    # ========== Household level (no alpha) ==========
    with plate('households', N_hh):
        if fixed_rho is None:
            rho_r = sample('rho_raw', dist.Normal(0, 1))
        if fixed_gamma is None:
            lg_r = sample('lgamma_raw', dist.Normal(0, 1))

    # ========== Bounded transforms (no alpha) ==========
    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_unbounded = mu_rho_C[hh_country_idx] + sig_rho_C[hh_country_idx] * rho_r
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        lg_unbounded = mu_lg_C[hh_country_idx] + sig_lg_C[hh_country_idx] * lg_r
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(lg_unbounded))

    # ========== Observation level (alpha = 0, no ao * sl term) ==========
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        # alpha = 0 → no adjustment: ao * sl = 0
        log_q10_adj = jnp.clip(cf_log_q10[idx], -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx], -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx], -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


nps_noalpha, hist_noalpha = run_variant('noalpha', model_main_noalpha, D)

# ---------- Compare ----------
print('\n' + '='*60)
print('  COMPARISON: main vs noalpha')
print('='*60)
print(f'  main    → best ELBO: {max(hist_main):.1f}  |  PPC max diff: see above')
print(f'  noalpha → best ELBO: {max(hist_noalpha):.1f}  |  PPC max diff: see above')

In [ ]:
# ============================================================
# Cell 4c: Country-level α variant
# ============================================================

_ckpt = 'country_alpha_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

RHO_LO, RHO_HI      = 0.1, 5.0
ALPHA_BOUND          = 1.5
GAMMA_LO, GAMMA_HI  = 0.1, 30.0

def model_country_alpha(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # ========== Learnable beta ==========
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # ========== Global hyperpriors ==========
    if fixed_rho is None:
        mu_rho_G  = sample('mu_rho_G', dist.Normal(0.0, 1.0))
        sig_rho_G = sample('sigma_rho_G', dist.HalfNormal(0.5))
    if fixed_gamma is None:
        mu_lg_G  = sample('mu_lgamma_G', dist.Normal(0.0, 1.0))
        sig_lg_G = sample('sigma_lgamma_G', dist.HalfNormal(0.5))

    # Alpha global prior
    mu_a_G  = sample('mu_alpha_G', dist.Normal(0.0, 1.0))
    sig_a_G = sample('sigma_alpha_G', dist.HalfNormal(0.5))

    # ========== Country level ==========
    with plate('countries', N_country):
        if fixed_rho is None:
            mu_rho_C_r = sample('mu_rho_C_raw', dist.Normal(0, 1))
            sig_rho_C  = sample('sigma_rho_C', dist.HalfNormal(0.3))
        if fixed_gamma is None:
            mu_lg_C_r = sample('mu_lgamma_C_raw', dist.Normal(0, 1))
            sig_lg_C  = sample('sigma_lgamma_C', dist.HalfNormal(0.3))

        # Alpha at COUNTRY level (not household)
        alpha_C_raw = sample('alpha_C_raw', dist.Normal(0, 1))

    if fixed_rho is None:
        mu_rho_C = deterministic('mu_rho_C', mu_rho_G + sig_rho_G * mu_rho_C_r)
    if fixed_gamma is None:
        mu_lg_C = deterministic('mu_lgamma_C', mu_lg_G + sig_lg_G * mu_lg_C_r)

    # Country-level alpha with bounded transform
    alpha_C_unbounded = mu_a_G + sig_a_G * alpha_C_raw
    alpha_C = deterministic('alpha_C', ALPHA_BOUND * jnp.tanh(alpha_C_unbounded))

    # ========== Household level (rho + gamma only, no alpha) ==========
    with plate('households', N_hh):
        if fixed_rho is None:
            rho_r = sample('rho_raw', dist.Normal(0, 1))
        if fixed_gamma is None:
            lg_r = sample('lgamma_raw', dist.Normal(0, 1))

    # Bounded transforms for rho and gamma at household level
    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_unbounded = mu_rho_C[hh_country_idx] + sig_rho_C[hh_country_idx] * rho_r
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        lg_unbounded = mu_lg_C[hh_country_idx] + sig_lg_C[hh_country_idx] * lg_r
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(lg_unbounded))

    # Alpha: household inherits its country's value
    alpha_i = deterministic('alpha_i', alpha_C[hh_country_idx])

    # ========== Observation level ==========
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        ao = alpha_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        log_q10_adj = jnp.clip(cf_log_q10[idx] - ao * sl, -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx] - ao * sl, -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx] - ao * sl, -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


nps_calpha, hist_calpha = run_variant('country_alpha', model_country_alpha, D)

# ---------- Three-way comparison ----------
print('\n' + '='*60)
print('  COMPARISON: main vs noalpha vs country_alpha')
print('='*60)
print(f'  main          → best ELBO: {max(hist_main):>12.1f}  |  params: 93,950')
print(f'  noalpha       → best ELBO: {max(hist_noalpha):>12.1f}  |  params: 62,634')
print(f'  country_alpha → best ELBO: {max(hist_calpha):>12.1f}  |  params: ~62,650')
print()
if 'alpha_C' in nps_calpha:
    ac = nps_calpha['alpha_C']
    print('Country-level alpha estimates:')
    for c in range(ac.shape[1]):
        m = ac[:, c].mean()
        s = ac[:, c].std()
        print(f'  country {c}: alpha = {m:.3f} ± {s:.3f}')

In [ ]:
# ============================================================
# Cell 4d: Nigeria-only — test alpha identifiability
# ============================================================

# --- Step 1: Find Nigeria and build filtered dataset ---
# First, figure out which country index is Nigeria
print("Country mapping:")
for c in range(D['N_country']):
    mask_c = np.array(D['hh_country_idx']) == c
    cname = hh_countries[np.where(mask_c)[0][0]]
    n = mask_c.sum()
    print(f"  {c}: {cname}  (n_hh={n})")

# Identify Nigeria index
nga_country_idx = None
for c in range(D['N_country']):
    mask_c = np.array(D['hh_country_idx']) == c
    cname = hh_countries[np.where(mask_c)[0][0]]
    if 'nigeria' in cname.lower() or 'nga' in cname.lower():
        nga_country_idx = c
        break

if nga_country_idx is None:
    # If auto-detect fails, print and let user set manually
    print("\nCould not auto-detect Nigeria. Set nga_country_idx manually:")
    print("  nga_country_idx = ???")
    raise ValueError("Set nga_country_idx manually above")

print(f"\nNigeria = country index {nga_country_idx}")

# --- Step 2: Filter data to Nigeria-only ---
hh_country = np.array(D['hh_country_idx'])
obs_hh = np.array(D['obs_hh_idx'])

# Nigerian households
nga_hh_mask = hh_country == nga_country_idx
nga_hh_old = np.where(nga_hh_mask)[0]                  # old hh indices
old_to_new_hh = -np.ones(D['N_hh'], dtype=int)
old_to_new_hh[nga_hh_old] = np.arange(len(nga_hh_old))

# Observations belonging to Nigerian households
nga_obs_mask = np.isin(obs_hh, nga_hh_old)
nga_obs_idx = np.where(nga_obs_mask)[0]

# Remap observation-level arrays
nga_obs_hh_new = old_to_new_hh[obs_hh[nga_obs_idx]]
nga_obs_cz = np.array(D['obs_cz_idx'])[nga_obs_idx]

# Remap zone indices to contiguous
nga_zones_old = np.unique(nga_obs_cz)
old_to_new_cz = -np.ones(int(np.array(D['obs_cz_idx']).max()) + 1, dtype=int)
old_to_new_cz[nga_zones_old] = np.arange(len(nga_zones_old))
nga_obs_cz_new = old_to_new_cz[nga_obs_cz]
N_nga_zones = len(nga_zones_old)

N_nga_hh = len(nga_hh_old)
N_nga_obs = len(nga_obs_idx)

print(f"Nigeria subset: {N_nga_hh} households, {N_nga_obs} observations, {N_nga_zones} zones")

# Build Nigeria-only D
D_nga = {
    'obs_action':       jnp.array(np.array(D['obs_action'])[nga_obs_idx]),
    'obs_hh_idx':       jnp.array(nga_obs_hh_new),
    'hh_country_idx':   jnp.zeros(N_nga_hh, dtype=int),     # single country → all 0
    'obs_cz_idx':       jnp.array(nga_obs_cz_new),
    'feasibility_mask': jnp.array(np.array(D['feasibility_mask'])[nga_zones_old]),
    'cf_log_q10':       jnp.array(np.array(D['cf_log_q10'])[nga_obs_idx]),
    'cf_log_q50':       jnp.array(np.array(D['cf_log_q50'])[nga_obs_idx]),
    'cf_log_q90':       jnp.array(np.array(D['cf_log_q90'])[nga_obs_idx]),
    'cf_sigma_log':     jnp.array(np.array(D['cf_sigma_log'])[nga_obs_idx]),
    'N_hh':             N_nga_hh,
    'N_country':        1,
    'N_obs':            N_nga_obs,
    'batch_size':       min(4096, N_nga_obs),
    'eps':              1.0,
    'fixed_rho':        None,
    'fixed_gamma':      None,
}

# --- Step 3: Model with single alpha (wide bound) ---
_ckpt = 'nga_alpha_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)

RHO_LO, RHO_HI      = 0.1, 5.0
ALPHA_BOUND          = 3.0       # wide — let the data decide
GAMMA_LO, GAMMA_HI  = 0.1, 30.0

def model_nga_alpha(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # Learnable beta
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # Single alpha for Nigeria (country-level, N_country=1)
    alpha_raw = sample('alpha_raw_global', dist.Normal(0.0, 1.0))
    alpha_val = deterministic('alpha_val', ALPHA_BOUND * jnp.tanh(alpha_raw))

    # Rho / gamma hyperpriors
    if fixed_rho is None:
        mu_rho  = sample('mu_rho', dist.Normal(0.0, 1.0))
        sig_rho = sample('sigma_rho', dist.HalfNormal(0.5))
    if fixed_gamma is None:
        mu_lg  = sample('mu_lgamma', dist.Normal(0.0, 1.0))
        sig_lg = sample('sigma_lgamma', dist.HalfNormal(0.5))

    # Household level
    with plate('households', N_hh):
        if fixed_rho is None:
            rho_r = sample('rho_raw', dist.Normal(0, 1))
        if fixed_gamma is None:
            lg_r = sample('lgamma_raw', dist.Normal(0, 1))

    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_unbounded = mu_rho + sig_rho * rho_r
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        lg_unbounded = mu_lg + sig_lg * lg_r
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(lg_unbounded))

    # Observations
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        log_q10_adj = jnp.clip(cf_log_q10[idx] - alpha_val * sl, -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx] - alpha_val * sl, -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx] - alpha_val * sl, -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


# --- Run ---
nps_nga, hist_nga = run_variant('nga_alpha', model_nga_alpha, D_nga)

# --- Print alpha result ---
if 'alpha_val' in nps_nga:
    av = nps_nga['alpha_val']
    print(f'\n*** Nigeria alpha = {av.mean():.4f} ± {av.std():.4f} ***')
    print(f'    (bound was ±{ALPHA_BOUND}, {"NOT binding" if abs(av.mean()) < ALPHA_BOUND * 0.9 else "BINDING!"})')
else:
    print('alpha_val not in samples, available:', list(nps_nga.keys()))

In [ ]:
# 模型跑完了，手动提取后验
from numpyro.infer import Predictive

_bound_nga = partial(model_nga_alpha, **{k: v for k, v in D_nga.items() if k in _STATIC_KEYS})
_dyn_nga   = {k: v for k, v in D_nga.items() if k not in _STATIC_KEYS}

# 从最后的 checkpoint 恢复 params
with open('nga_alpha_svi_checkpoint.pkl', 'rb') as f:
    p = pickle.load(f)

guide_nga = AutoNormal(_bound_nga)
svi_nga = SVI(_bound_nga, guide_nga, ClippedAdam(step_size=1e-4), Trace_ELBO())
state_nga = svi_nga.init(jax.random.PRNGKey(0), **_dyn_nga)
state_nga = jax.tree.unflatten(p['treedef'], [jnp.array(x) for x in p['flat_state']])
params_nga = svi_nga.get_params(state_nga)

pred_nga = Predictive(_bound_nga, guide=guide_nga, params=params_nga,
                       num_samples=1000, return_sites=['alpha_val', 'beta', 'rho_i', 'gamma_i'])
samples = pred_nga(jax.random.PRNGKey(99), **_dyn_nga)

av = np.array(samples['alpha_val'])
bv = np.array(samples['beta'])
print(f'Nigeria alpha = {av.mean():.4f} ± {av.std():.4f}')
print(f'Nigeria beta  = {bv.mean():.4f} ± {bv.std():.4f}')
print(f'Bound was ±3.0 → {"NOT binding" if abs(av.mean()) < 2.7 else "BINDING!"}')

rv = np.array(samples['rho_i']).mean(0)
gv = np.array(samples['gamma_i']).mean(0)
print(f'rho:   med={np.median(rv):.3f}  P5={np.percentile(rv,5):.3f}  P95={np.percentile(rv,95):.3f}')
print(f'gamma: med={np.median(gv):.3f}  P5={np.percentile(gv,5):.3f}  P95={np.percentile(gv,95):.3f}')

In [ ]:
# ============================================================
# Cell 4e: Hierarchical Stone-Geary
#   - alpha:  country-level only
#   - rho:    country mean + household offset
#   - gamma:  country mean + household offset
#   - beta:   learned (global)
#   Shrinkage: small countries (e.g. Tanzania n=121) borrow
#   strength from the global mean automatically.
# ============================================================

_ckpt = 'hier_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

# --- Bounded parameter ranges ---
RHO_LO, RHO_HI      = 0.1, 5.0
ALPHA_BOUND          = 3.0        # wide, Nigeria showed ~0.2 is typical
GAMMA_LO, GAMMA_HI  = 0.1, 30.0

def model_hier(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # =============================================
    #  Learnable beta (global)
    # =============================================
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # =============================================
    #  Global hyperpriors (group-level means & sds)
    # =============================================
    # -- rho --
    if fixed_rho is None:
        mu_rho_G      = sample('mu_rho_G', dist.Normal(0.0, 1.0))
        sigma_rho_G   = sample('sigma_rho_G', dist.HalfNormal(0.5))
        sigma_rho_W   = sample('sigma_rho_W', dist.HalfNormal(0.3))   # within-country sd

    # -- gamma --
    if fixed_gamma is None:
        mu_gamma_G    = sample('mu_gamma_G', dist.Normal(0.0, 1.0))
        sigma_gamma_G = sample('sigma_gamma_G', dist.HalfNormal(0.5))
        sigma_gamma_W = sample('sigma_gamma_W', dist.HalfNormal(0.3))  # within-country sd

    # -- alpha (country-level only, no within-country variation) --
    mu_alpha_G    = sample('mu_alpha_G', dist.Normal(0.0, 1.0))
    sigma_alpha_G = sample('sigma_alpha_G', dist.HalfNormal(0.5))

    # =============================================
    #  Country level (non-centered parameterization)
    # =============================================
    with plate('countries', N_country):
        # rho_c
        if fixed_rho is None:
            rho_c_raw = sample('rho_c_raw', dist.Normal(0, 1))

        # gamma_c
        if fixed_gamma is None:
            gamma_c_raw = sample('gamma_c_raw', dist.Normal(0, 1))

        # alpha_c
        alpha_c_raw = sample('alpha_c_raw', dist.Normal(0, 1))

    # Country-level deterministics (unbounded space)
    if fixed_rho is None:
        rho_c_unbounded = deterministic(
            'rho_c_unbounded', mu_rho_G + sigma_rho_G * rho_c_raw)
    if fixed_gamma is None:
        gamma_c_unbounded = deterministic(
            'gamma_c_unbounded', mu_gamma_G + sigma_gamma_G * gamma_c_raw)

    # Alpha: country-level, bounded
    alpha_c_unbounded = mu_alpha_G + sigma_alpha_G * alpha_c_raw
    alpha_c = deterministic('alpha_c', ALPHA_BOUND * jnp.tanh(alpha_c_unbounded))

    # =============================================
    #  Household level (offsets around country mean)
    # =============================================
    with plate('households', N_hh):
        if fixed_rho is None:
            rho_offset = sample('rho_offset', dist.Normal(0, 1))
        if fixed_gamma is None:
            gamma_offset = sample('gamma_offset', dist.Normal(0, 1))

    # Household-level parameters (bounded)
    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_i_unbounded = (rho_c_unbounded[hh_country_idx]
                           + sigma_rho_W * rho_offset)
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_i_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        gamma_i_unbounded = (gamma_c_unbounded[hh_country_idx]
                             + sigma_gamma_W * gamma_offset)
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(gamma_i_unbounded))

    # Alpha: household inherits country value (no hh-level variation)
    alpha_i = deterministic('alpha_i', alpha_c[hh_country_idx])

    # =============================================
    #  Observation level
    # =============================================
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        ao = alpha_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        log_q10_adj = jnp.clip(cf_log_q10[idx] - ao * sl, -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx] - ao * sl, -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx] - ao * sl, -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


# ---------- Sanity check ----------
print("Running sanity check (1 step)...")
D.pop('beta', None)
_bound = partial(model_hier, **{k: v for k, v in D.items() if k in _STATIC_KEYS})
_dyn   = {k: v for k, v in D.items() if k not in _STATIC_KEYS}
_guide = AutoNormal(_bound)
_svi   = SVI(_bound, _guide, ClippedAdam(step_size=1e-4, clip_norm=1.0), Trace_ELBO())
_state = _svi.init(jax.random.PRNGKey(0), **_dyn)
_state, _loss = jax.jit(_svi.update)(_state, **_dyn)
print(f"  1-step loss = {float(_loss):.1f}  {'OK' if not np.isnan(float(_loss)) else 'NaN!'}")
del _bound, _dyn, _guide, _svi, _state, _loss

# ---------- Full run ----------
nps_hier, hist_hier = run_variant('hier', model_hier, D)

# ---------- Country-level summary ----------
print('\n' + '='*60)
print('  COUNTRY-LEVEL PARAMETER ESTIMATES')
print('='*60)

hh_c = np.array(D['hh_country_idx'])
country_names = []
for c in range(D['N_country']):
    mask_c = hh_c == c
    country_names.append(hh_countries[np.where(mask_c)[0][0]])

# Alpha (country level — all hh in same country have same value)
if 'alpha_i' in nps_hier:
    ai = nps_hier['alpha_i']
    print(f'\n{"Country":<15s}  {"alpha":>10s}  {"rho_med":>10s}  {"rho_P5-P95":>15s}  {"gamma_med":>10s}  {"gamma_P5-P95":>15s}  {"n_hh":>6s}')
    print('-' * 85)
    for c in range(D['N_country']):
        mask_c = hh_c == c
        n = mask_c.sum()

        # Alpha: same for all hh in country
        first_hh = np.where(mask_c)[0][0]
        a_samples = ai[:, first_hh]
        a_str = f'{a_samples.mean():+.3f}±{a_samples.std():.3f}'

        # Rho: distribution across hh within country
        rho_hh = nps_hier['rho_i'][:, mask_c].mean(0)  # posterior mean per hh
        r_str = f'{np.median(rho_hh):.3f}'
        r_range = f'[{np.percentile(rho_hh,5):.2f}, {np.percentile(rho_hh,95):.2f}]'

        # Gamma
        g_hh = nps_hier['gamma_i'][:, mask_c].mean(0)
        g_str = f'{np.median(g_hh):.3f}'
        g_range = f'[{np.percentile(g_hh,5):.2f}, {np.percentile(g_hh,95):.2f}]'

        print(f'{country_names[c]:<15s}  {a_str:>10s}  {r_str:>10s}  {r_range:>15s}  {g_str:>10s}  {g_range:>15s}  {n:>6d}')

# Beta
if 'beta' in nps_hier:
    bv = nps_hier['beta']
    print(f'\nbeta (learned): {bv.mean():.4f} ± {bv.std():.4f}')

In [ ]:
# ============================================================
# Cell 4f: Hierarchical Stone-Geary — NO alpha
#   - rho:    country mean + household offset
#   - gamma:  country mean + household offset
#   - beta:   learned (global)
#   - alpha:  NONE
# ============================================================

_ckpt = 'hier_noalpha_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

RHO_LO, RHO_HI      = 0.1, 5.0
GAMMA_LO, GAMMA_HI  = 0.1, 30.0

def model_hier_noalpha(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # =============================================
    #  Learnable beta (global)
    # =============================================
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # =============================================
    #  Global hyperpriors
    # =============================================
    if fixed_rho is None:
        mu_rho_G      = sample('mu_rho_G', dist.Normal(0.0, 1.0))
        sigma_rho_G   = sample('sigma_rho_G', dist.HalfNormal(0.5))
        sigma_rho_W   = sample('sigma_rho_W', dist.HalfNormal(0.3))

    if fixed_gamma is None:
        mu_gamma_G    = sample('mu_gamma_G', dist.Normal(0.0, 1.0))
        sigma_gamma_G = sample('sigma_gamma_G', dist.HalfNormal(0.5))
        sigma_gamma_W = sample('sigma_gamma_W', dist.HalfNormal(0.3))

    # =============================================
    #  Country level (non-centered)
    # =============================================
    with plate('countries', N_country):
        if fixed_rho is None:
            rho_c_raw = sample('rho_c_raw', dist.Normal(0, 1))
        if fixed_gamma is None:
            gamma_c_raw = sample('gamma_c_raw', dist.Normal(0, 1))

    if fixed_rho is None:
        rho_c_unbounded = deterministic(
            'rho_c_unbounded', mu_rho_G + sigma_rho_G * rho_c_raw)
    if fixed_gamma is None:
        gamma_c_unbounded = deterministic(
            'gamma_c_unbounded', mu_gamma_G + sigma_gamma_G * gamma_c_raw)

    # =============================================
    #  Household level (offsets around country mean)
    # =============================================
    with plate('households', N_hh):
        if fixed_rho is None:
            rho_offset = sample('rho_offset', dist.Normal(0, 1))
        if fixed_gamma is None:
            gamma_offset = sample('gamma_offset', dist.Normal(0, 1))

    if fixed_rho is not None:
        rho_i = deterministic('rho_i', jnp.full(N_hh, fixed_rho))
    else:
        rho_i_unbounded = (rho_c_unbounded[hh_country_idx]
                           + sigma_rho_W * rho_offset)
        rho_i = deterministic('rho_i',
                              RHO_LO + (RHO_HI - RHO_LO) * jax.nn.sigmoid(rho_i_unbounded))

    if fixed_gamma is not None:
        gamma_i = deterministic('gamma_i', fixed_gamma)
    else:
        gamma_i_unbounded = (gamma_c_unbounded[hh_country_idx]
                             + sigma_gamma_W * gamma_offset)
        gamma_i = deterministic('gamma_i',
                                GAMMA_LO + (GAMMA_HI - GAMMA_LO) * jax.nn.sigmoid(gamma_i_unbounded))

    # =============================================
    #  Observation level (no alpha adjustment)
    # =============================================
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        ro = rho_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        log_q10_adj = jnp.clip(cf_log_q10[idx], -20.0, 20.0)
        log_q50_adj = jnp.clip(cf_log_q50[idx], -20.0, 20.0)
        log_q90_adj = jnp.clip(cf_log_q90[idx], -20.0, 20.0)
        q10 = jnp.exp(log_q10_adj)
        q50 = jnp.exp(log_q50_adj)
        q90 = jnp.exp(log_q90_adj)

        y1, y2, y3 = q10, 0.5 * (q10 + q50), q50
        y4, y5 = 0.5 * (q50 + q90), q90

        R = (0.10 * stone_geary_utility(y1, go, ro, eps)
           + 0.20 * stone_geary_utility(y2, go, ro, eps)
           + 0.40 * stone_geary_utility(y3, go, ro, eps)
           + 0.20 * stone_geary_utility(y4, go, ro, eps)
           + 0.10 * stone_geary_utility(y5, go, ro, eps))

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


# ---------- Sanity check ----------
print("Running sanity check (1 step)...")
D.pop('beta', None)
_bound = partial(model_hier_noalpha, **{k: v for k, v in D.items() if k in _STATIC_KEYS})
_dyn   = {k: v for k, v in D.items() if k not in _STATIC_KEYS}
_guide = AutoNormal(_bound)
_svi   = SVI(_bound, _guide, ClippedAdam(step_size=1e-4, clip_norm=1.0), Trace_ELBO())
_state = _svi.init(jax.random.PRNGKey(0), **_dyn)
_state, _loss = jax.jit(_svi.update)(_state, **_dyn)
print(f"  1-step loss = {float(_loss):.1f}  {'OK' if not np.isnan(float(_loss)) else 'NaN!'}")
del _bound, _dyn, _guide, _svi, _state, _loss

# ---------- Full run ----------
nps_hier_na, hist_hier_na = run_variant('hier_noalpha', model_hier_noalpha, D)

# ---------- Country-level summary ----------
print('\n' + '='*60)
print('  COUNTRY-LEVEL PARAMETER ESTIMATES (no alpha)')
print('='*60)

hh_c = np.array(D['hh_country_idx'])
country_names = []
for c in range(D['N_country']):
    mask_c = hh_c == c
    country_names.append(hh_countries[np.where(mask_c)[0][0]])

print(f'\n{"Country":<15s}  {"rho_med":>10s}  {"rho_P5-P95":>15s}  '
      f'{"gamma_med":>10s}  {"gamma_P5-P95":>15s}  {"n_hh":>6s}')
print('-' * 75)
for c in range(D['N_country']):
    mask_c = hh_c == c
    n = mask_c.sum()

    rho_hh = nps_hier_na['rho_i'][:, mask_c].mean(0)
    r_str = f'{np.median(rho_hh):.3f}'
    r_range = f'[{np.percentile(rho_hh,5):.2f}, {np.percentile(rho_hh,95):.2f}]'

    g_hh = nps_hier_na['gamma_i'][:, mask_c].mean(0)
    g_str = f'{np.median(g_hh):.3f}'
    g_range = f'[{np.percentile(g_hh,5):.2f}, {np.percentile(g_hh,95):.2f}]'

    print(f'{country_names[c]:<15s}  {r_str:>10s}  {r_range:>15s}  '
          f'{g_str:>10s}  {g_range:>15s}  {n:>6d}')

if 'beta' in nps_hier_na:
    bv = nps_hier_na['beta']
    print(f'\nbeta (learned): {bv.mean():.4f} ± {bv.std():.4f}')

In [ ]:
# ============================================================
# Cell 5: R2 — Downside Probability (λ, γ) Hierarchical
#   - lambda:  country mean + household offset
#   - gamma:   country mean + household offset
#   - beta:    learned (global)
#   - alpha:   NONE (not identifiable in joint estimation)
# ============================================================

_ckpt = 'R2_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)
    print(f'Deleted stale checkpoint: {_ckpt}')

# --- Bounded parameter ranges ---
LAMBDA_LO, LAMBDA_HI = 0.1, 200.0     # loss aversion weight
GAMMA_LO_R2, GAMMA_HI_R2 = 0.1, 30.0  # subsistence threshold (same data scale)

def model_R2(
    obs_action, obs_hh_idx, hh_country_idx, obs_cz_idx,
    feasibility_mask, cf_log_q10, cf_log_q50, cf_log_q90, cf_sigma_log,
    N_hh, N_country, N_obs, batch_size=4096, eps=1.0,
    fixed_rho=None, fixed_gamma=None,
):
    # =============================================
    #  Learnable beta (global)
    # =============================================
    log_beta = sample('log_beta', dist.Normal(1.0, 0.5))
    beta = deterministic('beta', jnp.exp(jnp.clip(log_beta, -2.0, 4.0)))

    # =============================================
    #  Global hyperpriors
    # =============================================
    # -- lambda --
    mu_lam_G      = sample('mu_lam_G', dist.Normal(0.0, 1.0))
    sigma_lam_G   = sample('sigma_lam_G', dist.HalfNormal(0.5))
    sigma_lam_W   = sample('sigma_lam_W', dist.HalfNormal(0.3))

    # -- gamma --
    mu_gamma_G    = sample('mu_gamma_G', dist.Normal(0.0, 1.0))
    sigma_gamma_G = sample('sigma_gamma_G', dist.HalfNormal(0.5))
    sigma_gamma_W = sample('sigma_gamma_W', dist.HalfNormal(0.3))

    # =============================================
    #  Country level (non-centered)
    # =============================================
    with plate('countries', N_country):
        lam_c_raw   = sample('lam_c_raw', dist.Normal(0, 1))
        gamma_c_raw = sample('gamma_c_raw', dist.Normal(0, 1))

    lam_c_unbounded = deterministic(
        'lam_c_unbounded', mu_lam_G + sigma_lam_G * lam_c_raw)
    gamma_c_unbounded = deterministic(
        'gamma_c_unbounded', mu_gamma_G + sigma_gamma_G * gamma_c_raw)

    # =============================================
    #  Household level (offsets around country mean)
    # =============================================
    with plate('households', N_hh):
        lam_offset   = sample('lam_offset', dist.Normal(0, 1))
        gamma_offset = sample('gamma_offset', dist.Normal(0, 1))

    # Bounded transforms
    lam_i_unbounded = (lam_c_unbounded[hh_country_idx]
                       + sigma_lam_W * lam_offset)
    lambda_i = deterministic('lambda_i',
                             LAMBDA_LO + (LAMBDA_HI - LAMBDA_LO) * jax.nn.sigmoid(lam_i_unbounded))

    gamma_i_unbounded = (gamma_c_unbounded[hh_country_idx]
                         + sigma_gamma_W * gamma_offset)
    gamma_i = deterministic('gamma_i',
                            GAMMA_LO_R2 + (GAMMA_HI_R2 - GAMMA_LO_R2) * jax.nn.sigmoid(gamma_i_unbounded))

    # =============================================
    #  Observation level (no alpha)
    # =============================================
    with plate('observations', N_obs, subsample_size=batch_size) as idx:
        la = lambda_i[obs_hh_idx[idx]][:, None]
        go = gamma_i[obs_hh_idx[idx]][:, None]

        sl = cf_sigma_log[idx]
        log_mu = jnp.clip(cf_log_q50[idx], -20.0, 20.0)
        mu_s = jnp.exp(log_mu)

        # Downside probability: P(Y < gamma)
        z = (jnp.log(go + 1.0) - log_mu) / jnp.maximum(sl, 0.01)
        R = mu_s - la * jax.scipy.stats.norm.cdf(z)

        mask   = feasibility_mask[obs_cz_idx[idx]]
        R      = _zscore_reward(R, mask)
        logits = jnp.where(mask, beta * R, -1e10)

        obs = obs_action[idx] if obs_action is not None else None
        sample('obs_action', dist.Categorical(logits=logits), obs=obs)


# ---------- Sanity check ----------
print("Running sanity check (1 step)...")
D.pop('beta', None)
_bound = partial(model_R2, **{k: v for k, v in D.items() if k in _STATIC_KEYS})
_dyn   = {k: v for k, v in D.items() if k not in _STATIC_KEYS}
_guide = AutoNormal(_bound)
_svi   = SVI(_bound, _guide, ClippedAdam(step_size=1e-4, clip_norm=1.0), Trace_ELBO())
_state = _svi.init(jax.random.PRNGKey(0), **_dyn)
_state, _loss = jax.jit(_svi.update)(_state, **_dyn)
print(f"  1-step loss = {float(_loss):.1f}  {'OK' if not np.isnan(float(_loss)) else 'NaN!'}")
del _bound, _dyn, _guide, _svi, _state, _loss

# ---------- Full run ----------
nps_R2, hist_R2 = run_variant('R2', model_R2, D)

# ---------- Country-level summary ----------
print('\n' + '='*60)
print('  COUNTRY-LEVEL PARAMETER ESTIMATES (R2: Downside Probability)')
print('='*60)

hh_c = np.array(D['hh_country_idx'])
country_names = []
for c in range(D['N_country']):
    mask_c = hh_c == c
    country_names.append(hh_countries[np.where(mask_c)[0][0]])

print(f'\n{"Country":<15s}  {"lambda_med":>10s}  {"lambda_P5-P95":>15s}  '
      f'{"gamma_med":>10s}  {"gamma_P5-P95":>15s}  {"n_hh":>6s}')
print('-' * 75)
for c in range(D['N_country']):
    mask_c = hh_c == c
    n = mask_c.sum()

    l_hh = nps_R2['lambda_i'][:, mask_c].mean(0)
    l_str = f'{np.median(l_hh):.3f}'
    l_range = f'[{np.percentile(l_hh,5):.2f}, {np.percentile(l_hh,95):.2f}]'

    g_hh = nps_R2['gamma_i'][:, mask_c].mean(0)
    g_str = f'{np.median(g_hh):.3f}'
    g_range = f'[{np.percentile(g_hh,5):.2f}, {np.percentile(g_hh,95):.2f}]'

    print(f'{country_names[c]:<15s}  {l_str:>10s}  {l_range:>15s}  '
          f'{g_str:>10s}  {g_range:>15s}  {n:>6d}')

if 'beta' in nps_R2:
    bv = nps_R2['beta']
    print(f'\nbeta (learned): {bv.mean():.4f} ± {bv.std():.4f}')

In [ ]:
# ============================================================
# Cell 6: R3 — Hierarchical Stone-Geary, fixed ρ=1.5 (γ only)
# ============================================================

_ckpt = 'R3_svi_checkpoint.pkl'
if os.path.exists(_ckpt):
    os.remove(_ckpt)

nps_R3, hist_R3 = run_variant('R3', model_hier_noalpha, {**D, 'fixed_rho': 1.5})

In [ ]:
print('='*60)
print('  COUNTRY-LEVEL ESTIMATES: R3 (fixed rho=1.5)')
print('='*60)

hh_c = np.array(D['hh_country_idx'])
print(f'\n{"Country":<15s}  {"gamma_med":>10s}  {"gamma_P5-P95":>15s}  {"n_hh":>6s}')
print('-'*55)
for c in range(D['N_country']):
    mask_c = hh_c == c
    n = mask_c.sum()
    cname = hh_countries[np.where(mask_c)[0][0]]

    g_hh = nps_R3['gamma_i'][:, mask_c].mean(0)
    g_str = f'{np.median(g_hh):.3f}'
    g_range = f'[{np.percentile(g_hh,5):.2f}, {np.percentile(g_hh,95):.2f}]'

    print(f'{cname:<15s}  {g_str:>10s}  {g_range:>15s}  {n:>6d}')

In [ ]:
# ============================================================
# Certainty Equivalent comparison across models
# ============================================================

def compute_country_CE(nps, D, model_name, fixed_rho=None):
    """Compute CE per country using posterior means of rho & gamma.
    CE satisfies U(CE) = E[U(Y)] where Y is actual consumption.
    For Stone-Geary: CE = gamma + E[(Y-gamma)^(1-rho)]^(1/(1-rho))
    """
    hh_c = np.array(D['hh_country_idx'])
    obs_hh = np.array(D['obs_hh_idx'])
    q50 = np.exp(np.array(D['cf_log_q50']))  # (N_obs, N_actions)

    # Posterior mean per household
    rho_hh = nps['rho_i'].mean(0) if 'rho_i' in nps else np.full(D['N_hh'], fixed_rho)
    gamma_hh = nps['gamma_i'].mean(0)

    results = []
    for c in range(D['N_country']):
        mask_c = hh_c == c
        cname = hh_countries[np.where(mask_c)[0][0]]
        n_hh = mask_c.sum()

        # Country median rho and gamma
        rho_c = np.median(rho_hh[mask_c])
        gamma_c = np.median(gamma_hh[mask_c])

        # Get consumption observations for this country's households
        hh_set = set(np.where(mask_c)[0])
        obs_mask = np.array([int(h) in hh_set for h in obs_hh])
        Y = q50[obs_mask].flatten()  # all consumption values
        Y = Y[Y > 0]  # drop zeros

        # E[U(Y)]
        eps = 1.0
        surplus = np.maximum(Y - gamma_c, eps)
        a = 1.0 - rho_c
        if abs(a) < 0.02:
            EU = np.mean(np.log(surplus))
        else:
            EU = np.mean((surplus**a - 1.0) / a)

        # Invert: CE = gamma + ((EU * a + 1)^(1/a))  if a != 0
        if abs(a) < 0.02:
            CE = gamma_c + np.exp(EU)
        else:
            inside = EU * a + 1.0
            if inside > 0:
                CE = gamma_c + inside ** (1.0 / a)
            else:
                CE = gamma_c + eps  # fallback

        results.append({
            'country': cname, 'rho': rho_c, 'gamma': gamma_c,
            'CE': CE, 'median_Y': np.median(Y), 'n_hh': n_hh,
        })

    return pd.DataFrame(results).sort_values('CE')


# Compute CE for each model
print('='*80)
print('  CERTAINTY EQUIVALENT RANKINGS')
print('='*80)

models = {}
try:
    models['hier_noalpha'] = compute_country_CE(nps_hier_na, D, 'hier_noalpha')
except:
    pass
try:
    models['R3 (rho=1.5)'] = compute_country_CE(nps_R3, D, 'R3', fixed_rho=1.5)
except:
    pass
try:
    models['hier+alpha'] = compute_country_CE(nps_hier, D, 'hier+alpha')
except:
    pass
try:
    models['country_alpha'] = compute_country_CE(nps_calpha, D, 'country_alpha')
except:
    pass

for name, df in models.items():
    print(f'\n--- {name} ---')
    df['rank'] = range(1, len(df)+1)
    print(df[['rank','country','CE','rho','gamma','median_Y','n_hh']].to_string(index=False))

# Compare rankings across models
if len(models) >= 2:
    print(f'\n\n{"="*80}')
    print('  RANKING STABILITY CHECK')
    print('='*80)
    names = list(models.keys())
    print(f'\n{"Country":<15s}', end='')
    for n in names:
        print(f'  {n:>15s}', end='')
    print()
    print('-' * (15 + 17 * len(names)))

    # Get all countries
    countries = models[names[0]]['country'].tolist()
    for ctry in sorted(countries):
        print(f'{ctry:<15s}', end='')
        for n in names:
            df = models[n]
            rank = df[df['country']==ctry]['rank'].values[0]
            print(f'  {rank:>15d}', end='')
        print()

    # Spearman correlation between first two models
    from scipy.stats import spearmanr
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            r1 = models[names[i]].set_index('country')['CE']
            r2 = models[names[j]].set_index('country')['CE']
            common = r1.index.intersection(r2.index)
            rho_s, p = spearmanr(r1[common].rank(), r2[common].rank())
            print(f'\nSpearman({names[i]} vs {names[j]}): ρ={rho_s:.3f}  p={p:.3f}')

In [ ]:
print('='*70)
print('  MODEL COMPARISON RANKING')
print('='*70)
print(f'{"Model":<25s}  {"Best ELBO":>12s}  {"PPC diff":>10s}  {"Params":>8s}')
print('-'*70)

results = [
    ('hier (rho+gamma)',       max(hist_hier_na), 'see above', 62630),
    ('hier+alpha',             max(hist_hier),    0.0650,      62630),
    ('country_alpha',          max(hist_calpha),  0.0642,      62650),
    ('main (hh-level alpha)',  max(hist_main),    0.0646,      93950),
    ('noalpha (flat)',         max(hist_noalpha), 0.0704,      62634),
    ('R2 (downside prob)',     max(hist_R2),      0.1863,      62614),
]

# Add R3 if it exists
try:
    results.append(('R3 (fixed rho=1.5)', max(hist_R3), 'see above', None))
except:
    pass

results.sort(key=lambda x: -x[1])  # best ELBO first

for i, (name, elbo, ppc, params) in enumerate(results, 1):
    ppc_str = f'{ppc:.4f}' if isinstance(ppc, float) else ppc
    p_str = f'{params:,}' if params else '?'
    print(f'{i}. {name:<23s}  {elbo:>12.1f}  {ppc_str:>10s}  {p_str:>8s}')